# 02 — Data Cleaning

Profiles raw CSVs using the EDA skill workflow, applies all cleaning rules, and creates the target variable `position_gained`.

**Skill used:** `.claude/skills/exploratory-data-analysis/SKILL.md`

**Inputs:** `data/pit_stops.csv`, `data/lap_times.csv`, `data/results.csv`  
**Outputs:** `data/cleaned/pit_stops_clean.csv`, `data/cleaned/lap_times_clean.csv`, `data/cleaned/results_clean.csv`, `data/eda_reports/*.md`

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
CLEANED = DATA / 'cleaned'
EDA_REPORTS = DATA / 'eda_reports'

CLEANED.mkdir(exist_ok=True)
EDA_REPORTS.mkdir(exist_ok=True)

---
## Step 1 — Profile raw data (EDA skill workflow)

Following the EDA skill: detect format → analyse → generate markdown report → save.

For each CSV we check: shape, dtypes, missing values, duplicates, and value distributions.

In [ ]:
from datetime import datetime

def eda_report(df: pd.DataFrame, name: str, filepath: Path) -> str:
    """Generate a markdown EDA report following the exploratory-data-analysis skill format."""
    ts = datetime.now().strftime('%Y-%m-%d %H:%M')
    size_kb = filepath.stat().st_size / 1024

    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
    missing_df = missing_df[missing_df['missing'] > 0]

    dupes = df.duplicated().sum()
    stats = df.describe(include='all').T

    lines = [
        f"# EDA Report: {name}",
        f"> Generated: {ts}",
        "",
        "## Basic Information",
        f"- **File:** `{filepath.name}`",
        f"- **Size:** {size_kb:.1f} KB",
        f"- **Format:** CSV (general scientific data)",
        "",
        "## Data Structure",
        f"- **Rows:** {len(df):,}",
        f"- **Columns:** {df.shape[1]}",
        f"- **Duplicate rows:** {dupes:,}",
        "",
        "### Column Types",
        df.dtypes.to_markdown(),
        "",
        "## Missing Values",
    ]

    if missing_df.empty:
        lines.append("No missing values.")
    else:
        lines.append(missing_df.to_markdown())

    lines += [
        "",
        "## Statistical Summary",
        stats.to_markdown(),
        "",
        "## Key Findings",
        f"- {len(df):,} total records across {df.shape[1]} columns",
        f"- {dupes:,} duplicate rows detected",
        f"- {len(missing_df)} columns contain missing values" if not missing_df.empty else "- No missing values",
        "",
        "## Recommendations",
        "- Apply domain-specific filters (safety car laps, DNFs, formation laps)",
        "- Standardise categorical labels before feature engineering",
        "- Validate join keys across tables (season + round + driver_id)",
    ]

    return "\n".join(lines)


for csv_name in ['pit_stops', 'lap_times', 'results']:
    path = DATA / f'{csv_name}.csv'
    df = pd.read_csv(path)
    report = eda_report(df, csv_name, path)
    out = EDA_REPORTS / f'{csv_name}_eda_report.md'
    out.write_text(report)
    print(f"Saved: {out.name}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")

---
## Step 2 — Load raw data

In [ ]:
races     = pd.read_csv(DATA / 'races.csv')
results   = pd.read_csv(DATA / 'results.csv')
pit_stops = pd.read_csv(DATA / 'pit_stops.csv')
lap_times = pd.read_csv(DATA / 'lap_times.csv')

print(f"races:     {races.shape}")
print(f"results:   {results.shape}")
print(f"pit_stops: {pit_stops.shape}")
print(f"lap_times: {lap_times.shape}")

---
## Step 3 — Clean: results

**Decision:** Remove DNF entries where `status` is not a finishing classification. DNF position changes are not meaningful — a driver classified P15 who retired on lap 3 did not 'finish' P15 due to strategy.

**Decision:** `position` is stored as a string in the Ergast API (can be `'R'` for retired). Cast to numeric; non-numeric values are DNFs.

In [ ]:
results_clean = results.copy()

# Cast position to numeric — non-numeric (R, D, E, W, F, N) become NaN
results_clean['position'] = pd.to_numeric(results_clean['position'], errors='coerce')

before = len(results_clean)
results_clean = results_clean.dropna(subset=['position'])
print(f"Removed {before - len(results_clean):,} DNF / non-classified rows")

# Drop duplicates
before = len(results_clean)
results_clean = results_clean.drop_duplicates()
print(f"Removed {before - len(results_clean):,} duplicate rows")

results_clean['position'] = results_clean['position'].astype(int)
print(f"results_clean: {results_clean.shape}")

---
## Step 4 — Clean: pit_stops

**Decision:** `duration` is stored as a string (`'mm:ss.sss'` or `'ss.sss'`). Convert to total seconds for numerical analysis.

**Decision:** Pit stop durations < 1.5 s or > 120 s are physically impossible — flag as anomalous and remove. These likely represent data entry errors or missed lap annotations.

In [ ]:
pit_clean = pit_stops.copy()

def duration_to_seconds(val: str) -> float:
    """Convert Ergast duration string to seconds."""
    try:
        val = str(val).strip()
        if ':' in val:
            parts = val.split(':')
            return float(parts[0]) * 60 + float(parts[1])
        return float(val)
    except Exception:
        return float('nan')

pit_clean['duration_s'] = pit_clean['duration'].apply(duration_to_seconds)

before = len(pit_clean)
pit_clean = pit_clean.dropna(subset=['duration_s'])
pit_clean = pit_clean[(pit_clean['duration_s'] >= 1.5) & (pit_clean['duration_s'] <= 120)]
print(f"Removed {before - len(pit_clean):,} rows with invalid pit durations")

# Only keep pit stops for drivers who finished the race
finishers = set(zip(results_clean['season'], results_clean['round'], results_clean['driver_id']))
pit_clean['_key'] = list(zip(pit_clean['season'], pit_clean['round'], pit_clean['driver_id']))
before = len(pit_clean)
pit_clean = pit_clean[pit_clean['_key'].isin(finishers)].drop(columns='_key')
print(f"Removed {before - len(pit_clean):,} pit stops for DNF drivers")

before = len(pit_clean)
pit_clean = pit_clean.drop_duplicates()
print(f"Removed {before - len(pit_clean):,} duplicate rows")

print(f"pit_clean: {pit_clean.shape}")

---
## Step 5 — Clean: lap_times

**Decision:** Remove formation laps (lap number = 0). Formation lap times are not representative of race pace.

**Decision:** Remove pit laps from lap time analysis. The lap on which a driver pits is artificially slow and distorts pace comparisons. Pit laps are identified by joining with pit_stops on (season, round, driver_id, lap).

**Decision:** Remove lap times with `time` = null or which cannot be parsed to milliseconds (these occur during safety car periods where timing was interrupted).

**Decision:** Keep only laps for drivers who finished the race.

In [ ]:
lap_clean = lap_times.copy()

# Remove formation laps
before = len(lap_clean)
lap_clean = lap_clean[lap_clean['lap'] > 0]
print(f"Removed {before - len(lap_clean):,} formation laps")

# Remove pit laps
pit_lap_keys = set(
    zip(pit_clean['season'], pit_clean['round'], pit_clean['driver_id'], pit_clean['lap'])
)
lap_clean['_key'] = list(zip(lap_clean['season'], lap_clean['round'], lap_clean['driver_id'], lap_clean['lap']))
before = len(lap_clean)
lap_clean = lap_clean[~lap_clean['_key'].isin(pit_lap_keys)].drop(columns='_key')
print(f"Removed {before - len(lap_clean):,} pit laps")

# Keep only finishers
lap_clean['_key'] = list(zip(lap_clean['season'], lap_clean['round'], lap_clean['driver_id']))
before = len(lap_clean)
lap_clean = lap_clean[lap_clean['_key'].isin(finishers)].drop(columns='_key')
print(f"Removed {before - len(lap_clean):,} laps for DNF drivers")

# Remove rows with missing time
before = len(lap_clean)
lap_clean = lap_clean.dropna(subset=['time'])
print(f"Removed {before - len(lap_clean):,} rows with missing lap time")

before = len(lap_clean)
lap_clean = lap_clean.drop_duplicates()
print(f"Removed {before - len(lap_clean):,} duplicate rows")

print(f"lap_clean: {lap_clean.shape}")

---
## Step 6 — Standardise tyre compound labels

**Decision:** Ergast does not carry tyre data directly — tyre compounds are sourced from the FastF1 cache and merged in the feature engineering notebook. For now we add a `compound` column placeholder. The standardisation mapping below will be applied in notebook 04.

The naming convention changed in 2019 (e.g. `ULTRASOFT` was retired). We map all historical names to five canonical labels:

| Raw | Canonical |
|---|---|
| SOFT, SUPERSOFT, ULTRASOFT, HYPERSOFT | Soft |
| MEDIUM | Medium |
| HARD | Hard |
| INTERMEDIATE | Intermediate |
| WET, FULL WET | Wet |

In [ ]:
COMPOUND_MAP = {
    'SOFT':          'Soft',
    'SUPERSOFT':     'Soft',
    'ULTRASOFT':     'Soft',
    'HYPERSOFT':     'Soft',
    'MEDIUM':        'Medium',
    'HARD':          'Hard',
    'INTERMEDIATE':  'Intermediate',
    'WET':           'Wet',
    'FULL WET':      'Wet',
}

VALID_COMPOUNDS = {'Soft', 'Medium', 'Hard', 'Intermediate', 'Wet'}

# The mapping will be applied in notebook 04 when FastF1 tyre data is joined.
# Exported here so feature engineering can import it.
print("Compound standardisation map defined.")
print(f"Canonical labels: {sorted(VALID_COMPOUNDS)}")

---
## Step 7 — Create target variable: `position_gained`

**Definition:** For each pit stop, compare the driver's track position immediately before pitting against their position in the lap after they rejoin.

**Iteration 1:** Attempted to use `results.grid` vs `results.position` — rejected because this compares start to finish, not pit-specific changes.

**Iteration 2:** Used lap_times position column. For each pit stop at lap `L`, take the driver's position on lap `L-1` (before pit) and lap `L+2` (after rejoining, allowing one lap to get up to speed). `position_gained = 1` if position improved by ≥1, else 0.

This is the measure we want: did this specific pit stop result in a net position gain?

In [ ]:
# Build a position lookup: (season, round, driver_id, lap) -> position
pos_lookup = lap_clean.set_index(
    ['season', 'round', 'driver_id', 'lap']
)['position'].to_dict()

def get_pos(season, rnd, driver, lap):
    return pos_lookup.get((season, rnd, driver, lap), np.nan)

pit_model = pit_clean.copy()
pit_model['pos_before'] = pit_model.apply(
    lambda r: get_pos(r['season'], r['round'], r['driver_id'], r['lap'] - 1), axis=1
)
pit_model['pos_after'] = pit_model.apply(
    lambda r: get_pos(r['season'], r['round'], r['driver_id'], r['lap'] + 2), axis=1
)

# Drop rows where we couldn't determine position before or after
before = len(pit_model)
pit_model = pit_model.dropna(subset=['pos_before', 'pos_after'])
print(f"Dropped {before - len(pit_model):,} rows with missing position data")

# position_change: positive = gained positions (lower number = further up)
pit_model['position_change'] = pit_model['pos_before'] - pit_model['pos_after']
pit_model['position_gained'] = (pit_model['position_change'] >= 1).astype(int)

print(f"\nTarget variable distribution:")
print(pit_model['position_gained'].value_counts())
print(f"\nClass balance: {pit_model['position_gained'].mean():.1%} gained position")

---
## Step 8 — Save cleaned data

In [ ]:
pit_model.to_csv(CLEANED / 'pit_stops_clean.csv', index=False)
lap_clean.to_csv(CLEANED / 'lap_times_clean.csv', index=False)
results_clean.to_csv(CLEANED / 'results_clean.csv', index=False)

print("Saved:")
print(f"  pit_stops_clean.csv  → {len(pit_model):,} rows")
print(f"  lap_times_clean.csv  → {len(lap_clean):,} rows")
print(f"  results_clean.csv    → {len(results_clean):,} rows")

# Verify target variable
assert pit_model['position_gained'].isnull().sum() == 0, "Nulls found in position_gained"
assert set(pit_model['position_gained'].unique()).issubset({0, 1}), "Non-binary values in position_gained"
print("\nAssertions passed: position_gained is binary with no nulls.")